In [1]:
import time
import numpy as np
import pickle
import pandas as pd
from scipy.stats import multivariate_normal, gaussian_kde
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score
from sklearn.feature_selection import mutual_info_regression

from scipy.integrate import quad

from scipy.stats import gaussian_kde

In [2]:
with open("../transformed_event_logs/BPIC_2017_all_train.pickle", "rb") as f:
    event_log = pickle.load(f)
event_log

/tmp/ipykernel_2891005/3913418224.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  event_log = pickle.load(f)


,Action_start,org:resource_start,concept:name,EventOrigin_start,EventID_start,lifecycle:transition_start,time:timestamp_start,case:LoanGoal_start,case:ApplicationType_start,case:concept:name,...,W_Shortened completion __resume,W_Shortened completion __schedule,W_Shortened completion __start,W_Shortened completion __suspend,W_Validate application__ate_abort,W_Validate application__complete,W_Validate application__resume,W_Validate application__schedule,W_Validate application__start,W_Validate application__suspend
1,Released,User_63,W_Call after offers__suspend,Workflow,Workitem_1000010198,suspend,2016-08-20 12:19:22.434000+00:00,Home improvement,New credit,Application_1930272371,...,0,0,0,0,0,0,0,0,0,0
2,Obtained,User_80,W_Call after offers__start,Workflow,Workitem_1000013868,start,2016-10-18 08:54:52.604000+00:00,Car,New credit,Application_1804686886,...,0,0,0,0,0,0,0,0,0,0
4,Obtained,User_114,W_Validate application__start,Workflow,Workitem_1000015916,start,2016-01-19 12:58:12.261000+00:00,Car,New credit,Application_704665572,...,0,0,0,0,0,0,0,1,1,0
5,Obtained,User_51,W_Call incomplete files__resume,Workflow,Workitem_1000016777,resume,2016-12-01 19:01:26.846000+00:00,Home improvement,New credit,Application_907188218,...,0,0,0,0,0,1,0,1,1,0
6,Obtained,User_5,W_Call after offers__resume,Workflow,Workitem_1000019350,resume,2016-02-13 15:34:25.947000+00:00,Home improvement,New credit,Application_2110398373,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619777,Released,User_19,W_Call after offers__suspend,Workflow,Workitem_999988201,suspend,2016-10-04 18:43:06.137000+00:00,Home improvement,New credit,Application_1551324804,...,0,0,0,0,0,0,0,0,0,0
619778,Obtained,User_28,W_Call incomplete files__resume,Workflow,Workitem_999990412,resume,2016-04-01 18:08:48.145000+00:00,Car,New credit,Application_883995052,...,0,0,0,0,1,0,0,1,1,1
619779,Released,User_51,W_Call after offers__suspend,Workflow,Workitem_999991648,suspend,2016-12-01 20:02:50.875000+00:00,Not speficied,New credit,Application_1000610355,...,0,0,0,0,0,0,0,0,0,0
619780,Created,User_118,W_Validate application__schedule,Workflow,Workitem_99999173,schedule,2016-04-13 07:49:14.822000+00:00,Not speficied,New credit,Application_52539020,...,0,0,0,0,0,0,0,1,0,0


In [3]:
list(event_log.columns)

['Action_start',
 'org:resource_start',
 'concept:name',
 'EventOrigin_start',
 'EventID_start',
 'lifecycle:transition_start',
 'time:timestamp_start',
 'case:LoanGoal_start',
 'case:ApplicationType_start',
 'case:concept:name',
 'case:RequestedAmount_start',
 'FirstWithdrawalAmount_start',
 'NumberOfTerms_start',
 'Accepted_start',
 'MonthlyCost_start',
 'Selected_start',
 'CreditScore_start',
 'OfferedAmount_start',
 'OfferID_start',
 'Action_complete',
 'org:resource_complete',
 'EventOrigin_complete',
 'EventID_complete',
 'lifecycle:transition_complete',
 'time:timestamp_complete',
 'case:LoanGoal_complete',
 'case:ApplicationType_complete',
 'case:RequestedAmount_complete',
 'FirstWithdrawalAmount_complete',
 'NumberOfTerms_complete',
 'Accepted_complete',
 'MonthlyCost_complete',
 'Selected_complete',
 'CreditScore_complete',
 'OfferedAmount_complete',
 'OfferID_complete',
 'duration',
 'duration_seconds',
 'duration_ms',
 'duration_hours',
 'seconds_in_day',
 'day_of_week',
 '

In [4]:
# numerical attributes : duration, seconds_in_day, day_in_week
numerical_attributes = [
    'duration_seconds',
    'seconds_in_day',
    'day_of_week',
    'case:RequestedAmount_start'
]


In [5]:
transformed_event_log = event_log.copy()

for num_attr in numerical_attributes:
    transformed_event_log[num_attr] = np.log(transformed_event_log[num_attr]+1)
    transformed_event_log[num_attr] = (transformed_event_log[num_attr] - transformed_event_log[num_attr].mean()) / transformed_event_log[num_attr].std()

In [6]:
transformed_event_log

,Action_start,org:resource_start,concept:name,EventOrigin_start,EventID_start,lifecycle:transition_start,time:timestamp_start,case:LoanGoal_start,case:ApplicationType_start,case:concept:name,...,W_Shortened completion __resume,W_Shortened completion __schedule,W_Shortened completion __start,W_Shortened completion __suspend,W_Validate application__ate_abort,W_Validate application__complete,W_Validate application__resume,W_Validate application__schedule,W_Validate application__start,W_Validate application__suspend
1,Released,User_63,W_Call after offers__suspend,Workflow,Workitem_1000010198,suspend,2016-08-20 12:19:22.434000+00:00,Home improvement,New credit,Application_1930272371,...,0,0,0,0,0,0,0,0,0,0
2,Obtained,User_80,W_Call after offers__start,Workflow,Workitem_1000013868,start,2016-10-18 08:54:52.604000+00:00,Car,New credit,Application_1804686886,...,0,0,0,0,0,0,0,0,0,0
4,Obtained,User_114,W_Validate application__start,Workflow,Workitem_1000015916,start,2016-01-19 12:58:12.261000+00:00,Car,New credit,Application_704665572,...,0,0,0,0,0,0,0,1,1,0
5,Obtained,User_51,W_Call incomplete files__resume,Workflow,Workitem_1000016777,resume,2016-12-01 19:01:26.846000+00:00,Home improvement,New credit,Application_907188218,...,0,0,0,0,0,1,0,1,1,0
6,Obtained,User_5,W_Call after offers__resume,Workflow,Workitem_1000019350,resume,2016-02-13 15:34:25.947000+00:00,Home improvement,New credit,Application_2110398373,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619777,Released,User_19,W_Call after offers__suspend,Workflow,Workitem_999988201,suspend,2016-10-04 18:43:06.137000+00:00,Home improvement,New credit,Application_1551324804,...,0,0,0,0,0,0,0,0,0,0
619778,Obtained,User_28,W_Call incomplete files__resume,Workflow,Workitem_999990412,resume,2016-04-01 18:08:48.145000+00:00,Car,New credit,Application_883995052,...,0,0,0,0,1,0,0,1,1,1
619779,Released,User_51,W_Call after offers__suspend,Workflow,Workitem_999991648,suspend,2016-12-01 20:02:50.875000+00:00,Not speficied,New credit,Application_1000610355,...,0,0,0,0,0,0,0,0,0,0
619780,Created,User_118,W_Validate application__schedule,Workflow,Workitem_99999173,schedule,2016-04-13 07:49:14.822000+00:00,Not speficied,New credit,Application_52539020,...,0,0,0,0,0,0,0,1,0,0


## Mutual Information: Discrete - Discrete

In [7]:
def MI_discrete_discrete(df, col, target_col):
    p_x = df[col].value_counts(normalize=True)
    p_y = df[target_col].value_counts(normalize=True)
    joint_counts = df.groupby([col, target_col]).size()
    p_xy = joint_counts / len(df)

    mi = 0.0
    for (x, y), pxy in p_xy.items():
        px = p_x[x]
        py = p_y[y]
        mi += pxy * np.log(pxy / (px * py))
    return mi


## Mutual Information: Discrete - Continuous

In [8]:
def MI_discrete_continuous(df, col, target_col='duration_seconds',
                gridsize=4096,  # power of two → cheap FFT in KDE
                eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *discrete* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    y_vals = df[target_col].to_numpy()
    y_min, y_max = y_vals.min(), y_vals.max()
    padding = 0.05 * (y_max - y_min)

    # Fixed grid avoids adaptive sampling where the KDE is unstable
    grid = np.linspace(y_min - padding, y_max + padding, gridsize)

    # p(y) and its log once
    kde_y      = gaussian_kde(y_vals)
    p_y_grid   = np.clip(kde_y(grid), eps, None)
    log_p_y    = np.log(p_y_grid)

    # p(x) for every category, with log once
    p_x        = df[col].value_counts(normalize=True)
    log_p_x    = np.log(p_x)

    # ------------------  Loop over each category of X  -----------------------
    mi = 0.0
    for x, log_px in log_p_x.items():
        subset = df.loc[df[col] == x, target_col]
        if len(subset.unique()) < 2:
            continue
        # KDE for p(y | x)  ≡  p(x,y) / p(x)
        kde_xy     = gaussian_kde(df.loc[df[col] == x, target_col])
        p_xy_grid = kde_xy(grid)

        p_joint_grid = p_xy_grid * p_x[x]
        p_joint_grid = np.clip(p_joint_grid, eps, None)

        log_p_joint = np.log(p_joint_grid)

        # integrand:  p_xy * (log p_xy − log p_x − log p_y)
        # do *all* log/exp algebra first, then multiply once
        integrand  = p_joint_grid * (log_p_joint - log_px - log_p_y)

        # trapz is deterministic, vectorised, and perfectly fine in 1‑D
        mi         += np.trapezoid(integrand, grid, dx=grid[1] - grid[0])

    return mi


def MI_discrete_continuous_2(df, col, target_col='duration_seconds',
                gridsize=4096,  # power of two → cheap FFT in KDE
                eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *discrete* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    y_vals = df[target_col].to_numpy()
    y_min, y_max = y_vals.min(), y_vals.max()
    padding = 0.05 * (y_max - y_min)

    # Fixed grid avoids adaptive sampling where the KDE is unstable
    grid = np.linspace(y_min - padding, y_max + padding, gridsize)

    # p(y) and its log once
    kde_y      = gaussian_kde(y_vals)
    p_y_grid   = np.clip(kde_y(grid), eps, None)
    log_p_y    = np.log(p_y_grid)

    # p(x) for every category, with log once
    p_x        = df[col].value_counts(normalize=True)
    log_p_x    = np.log(p_x)

    # ------------------  Loop over each category of X  -----------------------
    mi = 0.0
    for x, log_px in log_p_x.items():
        subset = df.loc[df[col] == x, target_col]
        if len(subset.unique()) < 2:
            continue
        # KDE for p(y | x)  ≡  p(x,y) / p(x)
        kde_xy     = gaussian_kde(df.loc[df[col] == x, target_col])
        p_xy_grid  = np.clip(kde_xy(grid), eps, None)
        log_p_xy   = np.log(p_xy_grid)

        # integrand:  p_xy * (log p_xy − log p_x − log p_y)
        # do *all* log/exp algebra first, then multiply once
        integrand  = p_x[x] * p_xy_grid * (log_p_xy - log_p_y)

        # trapz is deterministic, vectorised, and perfectly fine in 1‑D
        mi         += np.trapezoid(integrand, grid)
    return mi

def MI_discrete_continuous_3(df, col, target_col='duration_seconds',
                gridsize=4096,  # power of two → cheap FFT in KDE
                eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *discrete* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    y_vals = df[target_col].to_numpy()
    y_min, y_max = y_vals.min(), y_vals.max()
    padding = 0.05 * (y_max - y_min)

    # Fixed grid avoids adaptive sampling where the KDE is unstable
    grid = np.linspace(y_min - padding, y_max + padding, gridsize)

    # p(y) and its log once
    kde_y      = gaussian_kde(y_vals)
    p_y_grid   = np.clip(kde_y(grid), eps, None)
    log_p_y    = np.log(p_y_grid)

    # p(x) for every category, with log once
    p_x        = df[col].value_counts(normalize=True)
    log_p_x    = np.log(p_x)

    # ------------------  Loop over each category of X  -----------------------
    mi = 0.0
    for x, log_px in log_p_x.items():
        subset = df.loc[df[col] == x, target_col]
        if len(subset.unique()) < 2:
            continue
        # KDE for p(y | x)  ≡  p(x,y) / p(x)
        kde_xy     = gaussian_kde(df.loc[df[col] == x, target_col])
        p_xy_grid  = np.clip(kde_xy(grid), eps, None)
        log_p_xy   = np.log(p_xy_grid)

        # integrand:  p_xy * (log p_xy − log p_x − log p_y)
        # do *all* log/exp algebra first, then multiply once
        integrand  = p_xy_grid * (log_p_xy - log_p_y)

        # trapz is deterministic, vectorised, and perfectly fine in 1‑D
        mi         += p_x[x] * np.trapezoid(integrand, grid)
    return mi

## Mutual Information : Continuous - Continuous

In [9]:
def MI_continuous_continuous(df, col, target_col='duration_seconds',
                             gridsize=4096,  # power of two → cheap FFT in KDE
                             eps=1e-12):
    """
    Mutual information I(col ; target_col) for a *continuous* X=col and
    a *continuous* Y=target_col, using Gaussian KDE + log‑space integration.
    """
    # ------------------  Pre‑compute pieces that do not depend on x  ----------
    x_vals = df[col].to_numpy()
    y_vals = df[target_col].to_numpy()
    x_min, x_max = x_vals.min(), x_vals.max()
    y_min, y_max = y_vals.min(), y_vals.max()
    x_padding = 0 * 0.05 * (x_max - x_min)
    y_padding = 0 * 0.05 * (y_max - y_min)

    x_grid = np.linspace(x_min - x_padding, x_max + x_padding, gridsize)
    y_grid = np.linspace(y_min - y_padding, y_max + y_padding, gridsize)

    dx = x_grid[1] - x_grid[0]
    dy = y_grid[1] - y_grid[0]

    kde_x     = gaussian_kde(x_vals)
    kde_y     = gaussian_kde(y_vals)

    p_x_grid  = np.clip(kde_x(x_grid), eps, None)
    p_y_grid  = np.clip(kde_y(y_grid), eps, None)

    log_p_x   = np.log(p_x_grid)
    log_p_y    = np.log(p_y_grid)

    # Bivariate KDE
    kde_xy = gaussian_kde(np.vstack([x_vals, y_vals]))
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid, indexing='ij')  # Shape: (gridsize, gridsize)
    xy_samples = np.vstack([x_mesh.ravel(), y_mesh.ravel()])
    p_xy = np.clip(kde_xy(xy_samples), eps, None).reshape(gridsize, gridsize)
    log_p_xy = np.log(p_xy)

    # Compute MI: ∬ p(x, y) * (log p(x, y) - log p(x) - log p(y)) dx dy
    log_p_x_mesh = log_p_x[:, np.newaxis]  # shape (gridsize, 1)
    log_p_y_mesh = log_p_y[np.newaxis, :]  # shape (1, gridsize)

    integrand = p_xy * (log_p_xy - log_p_x_mesh - log_p_y_mesh)
    mi = np.sum(integrand) * dx * dy

    return mi

to_benchmark = lambda f, n : f(transformed_event_log, 'concept:name', gridsize=n)
l = [MI_discrete_continuous, MI_discrete_continuous_2, MI_discrete_continuous_3]


for i in [2**n for n in range(8, 14)]:
    for f in l:
        start_time = time.perf_counter()
        bf = to_benchmark(f, i)
        end_time = time.perf_counter()
        print(f"Function {f.__name__} with gridsize {i} took {end_time - start_time:.4f} seconds and returned {bf}")

### Test on synthetic data

In [10]:
def generate_bivariate_gaussian(n_samples, rho):
    """Generate bivariate Gaussian data with specified correlation."""
    mean = [0, 0]
    cov = [[1, rho], [rho, 1]]
    data = multivariate_normal.rvs(mean, cov, size=n_samples)
    return data

def true_mi_gaussian(rho):
    """Compute true MI for bivariate Gaussian variables."""
    return -0.5 * np.log(1 - rho**2)

test_data = generate_bivariate_gaussian(10000, 0.5)
test_df = pd.DataFrame(test_data, columns=['X', 'Y'])

In [11]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"n_neighbors={n}: {mutual_info_regression(test_df[['X']], test_df['Y'], n_neighbors=n)}")


n_neighbors=2: [0.14032262]
n_neighbors=4: [0.14510183]
n_neighbors=8: [0.14569119]
n_neighbors=16: [0.14781608]
n_neighbors=32: [0.14444014]
n_neighbors=64: [0.14571389]
n_neighbors=128: [0.14672823]
n_neighbors=256: [0.14648641]
n_neighbors=512: [0.14346189]
n_neighbors=1024: [0.13436055]
n_neighbors=2048: [0.11450642]
n_neighbors=4096: [0.07428317]


In [12]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    x_discrete = pd.cut(test_df['X'], bins=n, labels=False)
    y_discrete = pd.cut(test_df['Y'], bins=n, labels=False)

    # Compute MI
    print(f"bins={n}: {mutual_info_score(x_discrete, y_discrete)}")

bins=2: 0.05609224179913516
bins=4: 0.07792961948143504
bins=8: 0.12079698600430386
bins=16: 0.14506940724703044
bins=32: 0.17236076992775964
bins=64: 0.25310659446883055
bins=128: 0.479584500835639
bins=256: 1.0577112537632647
bins=512: 2.0799773020459535
bins=1024: 3.3172399210513754
bins=2048: 4.5970500288244445
bins=4096: 5.824560749747494


In [25]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"bins={n}: {MI_continuous_continuous(test_df, 'X', 'Y', n)}")

bins=2: 0.0013331438400678327
bins=4: 0.3174273501649583
bins=8: 0.1384186321447683
bins=16: 0.15060319587625312
bins=32: 0.14717438313824843
bins=64: 0.14690365549734893
bins=128: 0.14677369701761764
bins=256: 0.14670783954766894
bins=512: 0.14667470562979773
bins=1024: 0.14665808812377507
bins=2048: 0.1466497668505578
bins=4096: 0.14664560309979402


### Test on real world data

#### Continuous - Continuous

In [14]:
for n in [2, 4, 8, 16, 32, 64, 128, 256]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n)}")

bins=2: -0.006219673145877532
bins=4: 0.041912709129885935
bins=8: -0.1111848328586999
bins=16: -0.01337199651865282
bins=32: 0.0387453076153078
bins=64: 0.047367627078852836
bins=128: 0.04827994858992691
bins=256: 0.0492558780132657


In [15]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]:
    print(f"n_neighbors={n}: {mutual_info_regression(transformed_event_log[['seconds_in_day']], transformed_event_log['duration_seconds'], n_neighbors=n)}")

n_neighbors=2: [0.20325017]
n_neighbors=4: [0.1884694]
n_neighbors=8: [0.17791538]
n_neighbors=16: [0.17123444]
n_neighbors=32: [0.16374612]
n_neighbors=64: [0.15081969]
n_neighbors=128: [0.13508003]
n_neighbors=256: [0.11026815]
n_neighbors=512: [0.08604683]
n_neighbors=1024: [0.07348782]


In [16]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]:
    x_discrete = pd.cut(transformed_event_log['seconds_in_day'], bins=n, labels=False)
    y_discrete = pd.cut(transformed_event_log['duration_seconds'], bins=n, labels=False)

    # Compute MI
    print(f"bins={n}: {mutual_info_score(x_discrete, y_discrete)}")

bins=2: 3.372954906067575e-05
bins=4: 0.0001962677969524626
bins=8: 0.008846262545584758
bins=16: 0.03788353282090677
bins=32: 0.06580181320374269
bins=64: 0.0933262687993967
bins=128: 0.12631330837907997
bins=256: 0.16141824425422174
bins=512: 0.213868185291115
bins=1024: 0.3502523223692623


#### Discrete - Continuous

In [17]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"n_neighbors={n}: {MI_discrete_continuous(transformed_event_log, 'concept:name', 'case:RequestedAmount_start', n)}")

n_neighbors=2: 0.4874222083546823
n_neighbors=4: 0.25020828647269905
n_neighbors=8: 0.11391114055227677
n_neighbors=16: 0.16811848359127338
n_neighbors=32: 0.09173979661445067
n_neighbors=64: 0.08988058184843435
n_neighbors=128: 0.09051852093199751
n_neighbors=256: 0.09054978322346974
n_neighbors=512: 0.09055626398396581
n_neighbors=1024: 0.09055812990442094
n_neighbors=2048: 0.09055851462836051
n_neighbors=4096: 0.09055862645823715


In [18]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    y_discrete = pd.cut(transformed_event_log['case:RequestedAmount_start'], bins=n, labels=False)

    # Compute MI
    print(f"bins={n}: {mutual_info_score(transformed_event_log['concept:name'], y_discrete)}")

bins=2: 0.007523808123283973
bins=4: 0.009813720971477318
bins=8: 0.009921861351009496
bins=16: 0.011119774408357312
bins=32: 0.01226617382900907
bins=64: 0.012707342988396068
bins=128: 0.01340437559147339
bins=256: 0.014567193386044511
bins=512: 0.016421424352137837
bins=1024: 0.019267856265704417
bins=2048: 0.022749880557402694
bins=4096: 0.025074456869938408


In [19]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    print(f"n_neighbors={n}: {MI_discrete_continuous(transformed_event_log, 'concept:name', 'duration_seconds', n)}")

n_neighbors=2: 0.47044458019336094
n_neighbors=4: 1.0112018395128057
n_neighbors=8: 0.8328800367415644
n_neighbors=16: 0.7606072247553439
n_neighbors=32: 0.8413852130394021
n_neighbors=64: 0.848192884567055
n_neighbors=128: 0.8464943414745373
n_neighbors=256: 0.8466393192142624
n_neighbors=512: 0.8466987618547045
n_neighbors=1024: 0.878906647829458
n_neighbors=2048: 1.5559289751719378
n_neighbors=4096: 1.303865367763123


In [20]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    y_discrete = pd.cut(transformed_event_log['duration_seconds'], bins=n, labels=False)

    # Compute MI
    print(f"bins={n}: {mutual_info_score(transformed_event_log['concept:name'], y_discrete)}")

bins=2: 0.39933038756307315
bins=4: 0.6480195123847595
bins=8: 0.8831523283702105
bins=16: 1.0088929777738205
bins=32: 1.0404916776425455
bins=64: 1.0577451923714913
bins=128: 1.0685889408777394
bins=256: 1.0742063049868367
bins=512: 1.0805776509995797
bins=1024: 1.0897646089666886
bins=2048: 1.1059764998787165
bins=4096: 1.1426557370178663


#### Discrete - Discrete

In [21]:
MI_discrete_discrete(transformed_event_log, 'concept:name', 'org:resource_start')

np.float64(0.9226588415808465)

In [22]:
MI_discrete_discrete(transformed_event_log, 'concept:name', 'case:LoanGoal_start')

np.float64(0.01204242814457742)

In [23]:
MI_discrete_discrete(transformed_event_log, 'org:resource_start', 'case:LoanGoal_start')

np.float64(0.025963119650877137)

In [24]:
mutual_info_score(transformed_event_log['concept:name'], transformed_event_log['org:resource_start'])

0.9226588415808488